# CNN & TensorFlow 종합 실습 노트북

## 이 노트북이 다루는 것
한 노트북 안에 여러 모델/데이터셋이 단계별로 들어있음:

1. **[1] MNIST + FC network** — 가장 기본. 이미지를 1D 로 펴서 Dense 레이어만 사용
2. **[2] Fashion-MNIST + FC** — 같은 구조를 옷/신발 이미지에 적용 (더 어려운 태스크)
3. **[3] MNIST + CNN (v1)** — Conv+BatchNorm+Flatten+Dropout 기본 CNN
4. **[4] MNIST + CNN (v2)** — Conv(same padding)+BN+GAP+ReduceLROnPlateau 개선판
5. **[5] CIFAR-10 + CNN + TensorBoard** — 실제 컬러 이미지, 학습 로그 시각화
6. **[6] IMDB + Transformer Encoder** — 텍스트 감정 분류

## 공통적으로 배우는 패턴
- `to_categorical` 로 one-hot encoding (softmax + categorical_crossentropy 조합용)
- `EarlyStopping`, `ModelCheckpoint`, `ReduceLROnPlateau` 콜백
- `model.fit(validation_split=0.25)` 로 train/val 자동 분리
- 학습 후 `history.history` 로 loss 곡선 시각화
- FC → CNN (+ BatchNorm / GAP / padding='same') 으로 점진적 개선


---
# [1] MNIST 분류 — Fully Connected (FC) Network

손글씨 숫자 (0~9) 이미지 28×28 그레이스케일을 분류하는 가장 기본적인 태스크.

### 왜 FC 부터 시작하나
- 가장 단순한 모델 → 이후 CNN 과 성능 비교하기 좋은 baseline
- 이미지의 공간 구조 (2D) 를 무시하고 **784차원 벡터** 로 펴서 처리
- 이렇게 해도 MNIST 는 워낙 쉬운 태스크라 97~98% 정확도 나옴


## [1-1] 데이터 로드

Keras 에는 MNIST 가 내장되어 있어 한 줄로 받아올 수 있음.
- train 60000 장, test 10000 장
- 각 이미지는 28×28 의 uint8 배열 (0~255)
- 레이블은 0~9 정수


In [ ]:
# MNIST 데이터셋 로더 임포트
from tensorflow.keras.datasets import mnist


In [ ]:
# 첫 호출 시 인터넷에서 MNIST 를 다운로드받아 캐시함. 이후에는 캐시 사용
(x_train, y_train), (x_test, y_test) = mnist.load_data()


In [ ]:
# 크기 확인 — train 60000, test 10000, 각 (28, 28)
print("train size:", x_train.shape[0])
print("test size:", x_test.shape[0])
print(x_train.shape)   # (60000, 28, 28)
print(x_test.shape)    # (10000, 28, 28)


## [1-2] 샘플 이미지 시각화

실제로 어떻게 생긴 이미지인지 눈으로 확인.
`cmap='gray'` 는 값이 클수록 밝게 (보통), `cmap='Greys'` 는 값이 클수록 어둡게 (역전).
MNIST 는 배경이 검고 글씨가 흰 이미지라서 'gray' 쪽이 더 자연스러움.


In [ ]:
import matplotlib.pyplot as plt

# 첫 번째 샘플 — 검정 배경에 흰 숫자로 표시
plt.imshow(x_train[0], cmap='gray')
plt.show()

# 마지막 샘플 — cmap 을 바꿔서 흰 배경에 검은 숫자로
plt.imshow(x_train[59999], cmap='Greys')
plt.show()


## [1-3] raw 픽셀 값 확인

이미지를 숫자로 직접 출력해서 "정말 0~255 값이구나" 를 확인.
`%-4s` 포맷은 4자리로 왼쪽 정렬 — 숫자 행렬이 격자처럼 보이게 하는 트릭.


In [ ]:
import sys

# x_test[0] 은 (28, 28) — 바깥 for 가 행, 안쪽 for 가 열
for x in x_test[0]:
    for i in x:
        sys.stdout.write("%-4s" % i)   # 4자리 폭으로 숫자 출력
    sys.stdout.write('\n')


## [1-4] 이미지 펴기 (flatten)

FC network 는 1D 벡터만 입력으로 받음 → `(28, 28)` 을 `(784,)` 로 reshape.

### 왜 `28 * 28`?
`28 * 28 = 784` 픽셀. 공간 정보 (어느 픽셀이 이웃이었는지) 는 이 과정에서 사라짐.
CNN 은 이 정보를 보존하므로 더 잘 됨 → 이후 섹션에서 등장.


In [ ]:
# import 오타가 원본에 있지만 여기선 그냥 유지 (npm 은 사용 안 됨, 아래에 np 로 다시 import)
import numpy as npm

# (60000, 28, 28) → (60000, 784)
x_train = x_train.reshape(x_train.shape[0], 28 * 28)
x_test = x_test.reshape(x_test.shape[0], 28 * 28)

print(x_test[0])


## [1-5] Input scaling (정규화)

### 왜 255 로 나누나
- 원본 픽셀 값: 0~255 정수
- 신경망 학습은 **작은 범위 [0, 1] 이나 [-1, 1]** 에서 훨씬 안정적
  - gradient 크기가 일정해지고, 초기 weight 스케일과 궁합이 맞음
- `/255` 로 [0, 1] 로 맞춰줌 → 학습 속도 ↑, local minima 탈출 용이
- `astype('float32')` 는 먼저 해줘야 정수 나눗셈이 안 일어남


In [ ]:
# [1-5] input scaling — [0, 255] → [0, 1]
x_train = x_train.astype('float32')
x_train = x_train / 255
x_test  = x_test.astype('float32')
x_test  = x_test / 255


## [1-6] One-hot encoding

### 왜 one-hot?
- 레이블 `3` 을 정수 그대로 쓰면 softmax 출력 (10차원 확률벡터) 과 형태가 안 맞음
- 대신 `3 → [0,0,0,1,0,0,0,0,0,0]` 처럼 10차원 binary vector 로 변환
- 그러면 `categorical_crossentropy` 로 "예측 확률분포 vs 정답 분포" 직접 비교 가능

### 대안: `sparse_categorical_crossentropy`
- 정수 레이블을 그대로 쓰고 싶다면 loss 를 이걸로 바꾸면 됨 (결과 동일)
- 여기서는 전통적인 one-hot + categorical 방식을 씀


In [ ]:
from tensorflow.keras.utils import to_categorical

# 10 은 클래스 개수 (0~9) — 각 레이블이 10차원 벡터로 변환됨
y_train = to_categorical(y_train, 10)
y_test  = to_categorical(y_test, 10)

print('the first sample class:', y_train[0])   # e.g. [0,0,0,0,0,1,0,0,0,0] (=5)
print('the first sample class:', y_test[0])


## [1-7] FC 모델 정의

구조:
- **Input**: 784차원 벡터 (펴진 이미지)
- **Hidden**: Dense(512, relu) — 학습 가능한 특징 추출
- **Output**: Dense(10, softmax) — 10 클래스 확률

### 왜 512 뉴런?
관례적인 크기. 너무 작으면 표현력 부족, 너무 크면 과적합.
MNIST 는 쉬운 태스크라 이 정도면 충분.

### 왜 ReLU?
- `max(0, x)` — 양수는 통과, 음수는 0
- vanishing gradient 가 거의 없음 (양수 쪽 gradient = 1)
- Sigmoid/Tanh 보다 학습 빠름 → 현대 딥러닝 기본 활성화 함수

### 왜 softmax + categorical_crossentropy
- softmax: 출력을 합이 1인 확률분포로 변환
- categorical_crossentropy: 예측 확률과 정답 one-hot 의 크로스엔트로피
- 이 조합은 multi-class 분류의 정석


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

model = Sequential()
model.add(Input(shape=(784,)))                     # 입력 명시
model.add(Dense(512, activation='relu'))           # 은닉층
model.add(Dense(10, activation='softmax'))         # 출력층 (10 클래스 확률)

# Adam: 거의 모든 상황에서 잘 되는 기본 옵티마이저 (momentum + adaptive LR)
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy'],
)


## [1-8] 학습 + 콜백

### `ModelCheckpoint`
- 매 epoch 마다 val_loss 를 보고 **이전보다 낮아졌을 때만** 가중치 저장
- `save_best_only=True` → 최종적으로 파일엔 "제일 좋았던 epoch" 의 가중치만 남음

### `EarlyStopping(patience=10)`
- 10 epoch 동안 val_loss 가 개선되지 않으면 학습 중단
- 쓸데없이 오래 학습하다가 과적합되는 것을 방지

### `validation_split=0.25`
- train 데이터의 25% 를 자동으로 validation 으로 분리
- `validation_data=...` 를 따로 주지 않아도 됨

### `batch_size=200`
- 한 번에 200 샘플씩 묶어서 gradient 계산
- MNIST 에서 흔히 쓰는 크기 (너무 크면 메모리↑ 일반화↓, 너무 작으면 학습 불안정)

### `verbose=0`
- 진행상황 출력 안 함 (깔끔하게). 디버깅 시엔 `verbose=1` 로 바꿔서 써보기


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

modelpath = './models/cnn1_best.keras'

checkpointer = ModelCheckpoint(
    filepath=modelpath,
    monitor='val_loss',
    verbose=1,
    save_best_only=True,         # val_loss 가 개선될 때만 저장
)

early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=10,                 # 10 epoch 동안 개선 없으면 중단
)

# 학습 시작 — 최대 30 epoch, EarlyStopping 이 일찍 멈출 수 있음
history = model.fit(
    x_train, y_train,
    validation_split=0.25,       # train 의 25% 를 val 로 사용
    epochs=30,
    batch_size=200,
    verbose=0,
    callbacks=[early_stopping_callback, checkpointer],
)

# 최종 test 정확도 확인
test_acc = model.evaluate(x_test, y_test, verbose=0)[1]
print(f"\n Test accuracy: {test_acc:.4f}")


## [1-9] 학습 곡선 시각화

### 왜 두 곡선을 같이 보나
- **Train loss** 만 계속 감소 + **val loss** 는 어느 순간부터 증가 → **과적합** 신호
- 둘 다 비슷하게 감소 → 건강한 학습
- val loss 가 학습 후반에 튀면 EarlyStopping 의 `patience` 값이 적절한지 점검


In [ ]:
import numpy as np

y_vloss = history.history['val_loss']     # validation loss 기록
y_loss  = history.history['loss']         # training loss 기록
x_len   = np.arange(len(y_loss))          # epoch 축

plt.plot(x_len, y_vloss, marker='.', c='red',  label='Validation Loss')
plt.plot(x_len, y_loss,  marker='.', c='blue', label='Train Loss')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()


---
# [2] Fashion-MNIST — 같은 구조로 더 어려운 태스크

### Fashion-MNIST 가 MNIST 보다 어려운 이유
- 28×28 그레이스케일 이미지, 10 클래스 — 크기/형식은 똑같지만
- 내용이 **옷, 신발, 가방** 등 → 클래스 간 시각적 차이가 더 미묘
  (예: 셔츠 vs 티셔츠 vs 풀오버 — 실제로 사람도 헷갈림)
- 같은 FC(784 → 512 → 10) 로는 MNIST 98% vs Fashion 85~88% 수준

### 이 셀에서 달라지는 것
- 데이터셋만 `fashion_mnist.load_data()` 로 바뀜
- 나머지 전처리/모델/학습 파이프라인은 **완전히 동일**
- → "데이터가 바뀌어도 같은 코드가 먹힌다" 는 것을 시연


In [ ]:
# [2] Fashion-MNIST — FC network (동일 구조)
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import os

# 1. Load dataset — MNIST 와 완전 동일 형식
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# 2. Reshape & normalize — 펴기 + [0,1] 스케일
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 784).astype('float32') / 255.0

# 3. One-hot encoding
y_train = to_categorical(y_train, 10)
y_test  = to_categorical(y_test, 10)

# 4. Build model — MNIST 때와 동일
model = Sequential([
    Input(shape=(784,)),
    Dense(512, activation='relu'),
    Dense(10, activation='softmax'),
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy'],
)

# 5. Callbacks
os.makedirs('./models', exist_ok=True)   # 저장 폴더 미리 생성 (없으면 에러)
modelpath = './models/cnn1_best.keras'

checkpointer = ModelCheckpoint(
    filepath=modelpath, monitor='val_loss',
    verbose=1, save_best_only=True,
)
early_stopping = EarlyStopping(monitor='val_loss', patience=10)

# 6. Train model
history = model.fit(
    X_train, y_train,
    validation_split=0.25,
    epochs=30, batch_size=200,
    verbose=0,
    callbacks=[early_stopping, checkpointer],
)

# 7. Evaluate — MNIST 보다 정확도가 분명히 떨어짐을 확인
test_acc = model.evaluate(X_test, y_test, verbose=0)[1]
print(f"\nTest accuracy: {test_acc:.4f}")

# 8. Plot loss curves
y_vloss = history.history['val_loss']
y_loss  = history.history['loss']
x_len   = np.arange(len(y_loss))
plt.plot(x_len, y_vloss, marker='.', c='red',  label='Validation Loss')
plt.plot(x_len, y_loss,  marker='.', c='blue', label='Train Loss')
plt.legend(); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.show()


---
# [3] MNIST — CNN (Convolutional Neural Network) v1

### 왜 CNN?
FC 는 이미지를 1D 벡터로 펴기 때문에 **공간 정보가 소실**됨.
CNN 은 2D 구조를 유지하면서 "인접 픽셀끼리 계산" 하기 때문에:
- 파라미터 수가 훨씬 적어도 됨 (필터 하나를 이미지 전체에 공유 = weight sharing)
- translation invariance — 숫자가 이미지 중앙에 있든 왼쪽에 있든 비슷하게 인식
- **이미지 태스크에서는 거의 항상 FC 보다 잘 됨**

### 이 모델 구조
```
Input (28, 28, 1)
 → Conv2D(32, 3x3, ReLU) → BatchNorm
 → Conv2D(64, 3x3, ReLU) → BatchNorm
 → MaxPooling2D(2x2)
 → Flatten
 → Dense(128, ReLU) → Dropout(0.3)
 → Dense(10, softmax)
```

### 왜 BatchNormalization?
- 각 배치의 활성화를 **평균 0, 분산 1** 로 정규화
- 학습 안정성 ↑, 더 큰 learning rate 사용 가능
- 약한 regularization 효과도 있음

### 왜 Dropout?
- 학습 중 **일정 비율의 뉴런을 랜덤으로 끔**
- 특정 뉴런에 과도하게 의존하는 걸 방지 → 과적합 ↓
- test 시에는 자동으로 꺼지고 모든 뉴런을 사용


In [ ]:
# [3] MNIST — CNN v1 (Conv + BN + Flatten + Dropout)
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Load dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# 2. Reshape & normalize — CNN 은 2D 구조 유지, 채널 차원 추가 필요
#    (28, 28) → (28, 28, 1)  — 그레이스케일이라 채널 1
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# 3. One-hot encoding
y_train = to_categorical(y_train, 10)
y_test  = to_categorical(y_test, 10)

# 4. Build CNN model with BatchNorm
model = Sequential([
    Input(shape=(28, 28, 1)),

    # 첫 Conv: 32 필터 — 가장자리/간단한 에지 검출
    Conv2D(32, (3, 3), activation='relu'),
    BatchNormalization(),                    # 활성화 분포 정규화

    # 두 번째 Conv: 64 필터 — 더 복잡한 패턴
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),

    MaxPooling2D((2, 2)),                    # 공간 해상도 절반, 계산량 ↓, 일부 위치 불변성 ↑
    Flatten(),                                # FC 로 넘기기 위해 1D 로 펼침

    Dense(128, activation='relu'),
    Dropout(0.3),                            # 30% 랜덤 off — 과적합 방지

    Dense(10, activation='softmax'),
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy'],
)

# 5. Callbacks
os.makedirs('./models', exist_ok=True)
modelpath = './models/cnn2_best.keras'

checkpointer = ModelCheckpoint(
    filepath=modelpath, monitor='val_loss',
    verbose=1, save_best_only=True,
)
early_stopping = EarlyStopping(monitor='val_loss', patience=10)

# 6. Train
history = model.fit(
    X_train, y_train,
    validation_split=0.25,
    epochs=30, batch_size=200,
    verbose=1,
    callbacks=[early_stopping, checkpointer],
)

# 7. Evaluate — FC 보다 정확도가 올라가는지 확인 (보통 99%+)
test_acc = model.evaluate(X_test, y_test, verbose=0)[1]
print(f"\nTest accuracy: {test_acc:.4f}")

# 8. Plot loss
y_vloss = history.history['val_loss']
y_loss  = history.history['loss']
x_len   = np.arange(len(y_loss))
plt.plot(x_len, y_vloss, marker='.', c='red',  label='Validation Loss')
plt.plot(x_len, y_loss,  marker='.', c='blue', label='Train Loss')
plt.legend(); plt.grid()
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.show()


---
# [4] MNIST — CNN v2 (개선판: padding='same' + GAP + LR 스케줄)

### v1 대비 개선점
1. **`padding='same'`** — Conv 후에도 해상도 유지 (경계 zero-padding).
   더 깊게 쌓아도 spatial size 가 빨리 줄어들지 않음.
2. **`use_bias=False` + BatchNorm** — BN 이 shift 역할을 이미 하므로 bias 중복 제거.
   파라미터 수 살짝 감소, 효과는 거의 동일.
3. **Block 3개** — Conv→Conv→Pool→Conv 로 채널 수를 32→64→128 로 점진 증가.
4. **`GlobalAveragePooling2D` (GAP)** — Flatten+Dense(128) 대신 채널별 평균 하나로.
   - 파라미터 대폭 감소 (Flatten 은 수천~수만 weight 발생)
   - 과적합 억제 효과
   - 최근 CNN (ResNet 등) 에서 표준
5. **`ReduceLROnPlateau`** — val_loss 가 plateau 에 걸리면 LR 을 절반으로.
   - 학습 후반에 세밀한 조정 가능
6. **`restore_best_weights=True`** — EarlyStopping 후 "가장 좋았던 epoch" 의 가중치를 복원.
   - 체크포인트를 따로 로드하지 않아도 model 이 최적 상태가 됨
7. **batch_size 128** — 128~256 이 GAP + BN 조합에서 안정적


In [ ]:
# [4] MNIST — CNN v2 (BN + same padding + 3블록 + GAP + LR 스케줄)
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Load dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# 2. Reshape & normalize
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# 3. One-hot encoding
y_train = to_categorical(y_train, 10)
y_test  = to_categorical(y_test, 10)

# 4. Build CNN v2
model = Sequential([
    Input(shape=(28, 28, 1)),

    # Block 1 — 32 채널, same padding 으로 해상도 유지
    Conv2D(32, (3, 3), padding='same', activation='relu', use_bias=False),
    BatchNormalization(),

    # Block 2 — 64 채널, 뒤에 pool 로 해상도 절반
    Conv2D(64, (3, 3), padding='same', activation='relu', use_bias=False),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Block 3 — 128 채널, 깊은 특징
    Conv2D(128, (3, 3), padding='same', activation='relu', use_bias=False),
    BatchNormalization(),

    # GAP: 각 채널의 spatial 평균 → (batch, 128) 벡터
    # Flatten + Dense(128) 조합 대비 파라미터 수천~수만개 절감 + regularization
    GlobalAveragePooling2D(),

    Dropout(0.3),
    Dense(10, activation='softmax'),
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy'],
)

# 5. Callbacks
os.makedirs('./models', exist_ok=True)
checkpointer = ModelCheckpoint(
    filepath='./models/cnn2_best.keras',
    monitor='val_loss', verbose=1, save_best_only=True,
)
# ReduceLROnPlateau: 3 epoch 동안 val_loss 개선 없으면 LR *= 0.5
#   학습 후반에 loss 가 멈추면 LR 을 줄여서 더 세밀히 수렴
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, verbose=1,
)
# restore_best_weights: 학습 종료 후 자동으로 최적 가중치 복원
early_stopping = EarlyStopping(
    monitor='val_loss', patience=8,
    restore_best_weights=True, verbose=1,
)

# 6. Train — 권장 batch_size 128~256
history = model.fit(
    X_train, y_train,
    validation_split=0.25,
    epochs=30, batch_size=128,
    verbose=1,
    callbacks=[reduce_lr, early_stopping, checkpointer],
)

# 7. Evaluate
test_acc = model.evaluate(X_test, y_test, verbose=0)[1]
print(f"\nTest accuracy: {test_acc:.4f}")

# 8. Plot loss
y_vloss = history.history['val_loss']
y_loss  = history.history['loss']
x_len   = np.arange(len(y_loss))
plt.plot(x_len, y_vloss, marker='.', c='red',  label='Validation Loss')
plt.plot(x_len, y_loss,  marker='.', c='blue', label='Train Loss')
plt.legend(); plt.grid()
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.show()


---
# [5] CIFAR-10 + TensorBoard 시각화

### CIFAR-10 이 MNIST 보다 어려운 이유
- **컬러 (3채널)**, **32×32** 크기의 자연 이미지
- 10 클래스: 비행기/자동차/새/고양이/사슴/개/개구리/말/배/트럭
- 배경 잡음, 다양한 각도/조명 → 진짜 "컴퓨터 비전" 태스크
- FC 로는 50% 도 힘듦, 기본 CNN 으로 70~75%, 좋은 CNN 으로 85%+

### 이 모델 구조 — VGG 스타일 (Conv 두 번 쌓고 Pool)
```
Block 1: Conv(32) → Conv(32) → Pool → Dropout(0.25)
Block 2: Conv(64) → Conv(64) → Pool → Dropout(0.25)
Head:    Flatten → Dense(128) → Dropout(0.5) → Dense(10, softmax)
```

Conv 를 연속으로 두 개 쌓는 이유: 3×3 두 개 = 5×5 한 개의 receptive field 를 갖지만 파라미터가 더 적고 비선형성이 두 번 들어감.

### TensorBoard 란
- 학습 과정 (loss, accuracy, 가중치 분포, 그래프 등) 을 **웹 UI** 에서 실시간 시각화
- 셀이 실행된 후 터미널에서 `tensorboard --logdir ./logs/fit` 실행 → 브라우저에서 열람
- `histogram_freq=1` → 매 epoch 마다 각 레이어 weight 분포 기록


In [ ]:
# [5] CIFAR-10 + CNN + TensorBoard 로깅
import datetime
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
import os

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, TensorBoard

# 1) 데이터 로드 & 전처리 — CIFAR 는 shape=(32, 32, 3)
(x_train, y_train_raw), (x_test, y_test_raw) = cifar10.load_data()
x_train = x_train.astype("float32") / 255.0   # [0, 1] 스케일
x_test  = x_test.astype("float32")  / 255.0
y_train = to_categorical(y_train_raw, 10)
y_test  = to_categorical(y_test_raw, 10)

# 1-1) 샘플 확인 — 자연 이미지라 MNIST 와 달리 작지만 알아볼 수 있음
plt.imshow(x_train[0])
plt.title(f"Label(one-hot argmax): {y_train[0].argmax()}")
plt.axis("off")
plt.show()

# 2) 모델 정의 (VGG-style CNN)
model = Sequential([
    Input(shape=(32, 32, 3)),

    # Block 1
    Conv2D(32, (3, 3), activation="relu", padding="same"),
    Conv2D(32, (3, 3), activation="relu", padding="same"),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # Block 2 — 채널 수 증가로 더 복잡한 특징 학습
    Conv2D(64, (3, 3), activation="relu", padding="same"),
    Conv2D(64, (3, 3), activation="relu", padding="same"),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # Classifier head
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),                        # head 에서는 좀 더 강한 dropout
    Dense(10, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

# 3) 콜백 설정: TensorBoard + Checkpoint + EarlyStopping
os.makedirs("./logs/fit", exist_ok=True)
os.makedirs("./models", exist_ok=True)

# 실행 시점의 타임스탬프로 로그 디렉토리 생성 — 실행마다 분리되어 비교 가능
log_dir = "./logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_cb = TensorBoard(log_dir=log_dir, histogram_freq=1)  # 매 epoch weight 히스토그램 기록

ckpt_path = "./models/cifar10_cnn_best.keras"
checkpoint_cb = ModelCheckpoint(
    filepath=ckpt_path, monitor="val_loss",
    save_best_only=True, verbose=1,
)
earlystop_cb = EarlyStopping(
    monitor="val_loss", patience=10,
    restore_best_weights=True,           # 자동 최적 가중치 복원
)

# 4) 학습
history = model.fit(
    x_train, y_train,
    validation_split=0.25,
    epochs=30, batch_size=200,
    callbacks=[tensorboard_cb, checkpoint_cb, earlystop_cb],
    verbose=1,
)

# 5) 평가
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f} | Test loss: {test_loss:.4f}")

# TensorBoard 열기 (터미널에서):
#   tensorboard --logdir ./logs/fit
# 주피터 안에서는:  %load_ext tensorboard  →  %tensorboard --logdir ./logs/fit


---
# [6] IMDB 감정 분류 — Transformer Encoder

### 태스크
영화 리뷰 (영어 텍스트) → positive(1) / negative(0) 이진 분류

### 왜 Transformer?
- RNN/LSTM 은 순차적으로 처리 → 긴 문장에서 느리고 long-range dependency 약함
- Transformer 는 **전체 단어들이 서로를 바로 참조** (self-attention) → 병렬 + 장거리 의존성 ↑
- 2017 논문 "Attention Is All You Need" 이후 NLP 의 표준

### 이 파이프라인
```
text → TextVectorization (string → int 시퀀스)
     → Embedding (int → 256차원 vector)
     → TransformerEncoder (+ positional embedding) × 1
     → GlobalMaxPooling1D (시퀀스 → 벡터 1개)
     → Dropout → Dense(1, sigmoid)
```

### Positional Embedding 이 왜 필요?
- Transformer 는 attention 이 **순서를 모름** (위치 정보 없이 '단어 집합' 으로 봄)
- "나는 너를 좋아해" vs "너는 나를 좋아해" 가 구분 안 됨
- → 각 위치에 **학습 가능한 vector** 를 더해서 순서 정보 주입 (`layers.Embedding(max_length, embed_dim)`)

### 구조 요약
- `TransformerEncoder` 클래스 하나 안에 Multi-Head Attention + Dense FFN + 2개의 LayerNorm
- Residual connection 2개 (attention 블록, FFN 블록)
- 이게 Transformer 의 기본 "블록"


## [6-1] 데이터 준비 — IMDB 다운로드 및 분리

### 주의: shell 명령은 주석 처리됨
`!curl`, `!tar`, `!rm` 은 Jupyter 에서만 돌고 파이썬 파서는 못 읽음.
노트북에서 직접 실행하려면 `#` 을 제거하세요.

### 디렉토리 구조
```
aclImdb/
├── train/{pos, neg}/*.txt
├── test/{pos, neg}/*.txt
└── val/{pos, neg}/*.txt    (원본엔 없고, 코드가 train 20% 를 빼서 생성)
```

### 왜 random.Random(1337) 로 섞나
- 고정된 시드 → 매번 같은 파일이 validation 으로 빠짐 → 실험 재현성


In [ ]:
# [6-1] IMDB 감정 분류 — Transformer Encoder
import os, pathlib, shutil, random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

batch_size = 32

# 데이터 디렉토리 경로 설정
base_dir  = pathlib.Path("aclImdb")
val_dir   = base_dir / "val"
train_dir = base_dir / "train"

# IMDB 리뷰 데이터셋 다운로드 및 압축 해제 (주피터에서 수동 실행)
# !rm -r aclImdb
# !curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
# !tar -xf aclImdb_v1.tar.gz
# !rm -r aclImdb/train/unsup   # unsup(라벨 없음) 폴더 제거 — 지도학습엔 불필요

# train → train/val 분리 (원본엔 val 없음)
for category in ("neg", "pos"):
    os.makedirs(val_dir / category)
    files = os.listdir(train_dir / category)
    random.Random(1337).shuffle(files)           # 재현성을 위한 고정 시드
    num_val_samples = int(0.2 * len(files))      # 20% 를 val 로
    val_files = files[-num_val_samples:]
    for fname in val_files:
        shutil.move(train_dir / category / fname, val_dir / category / fname)

# 텍스트 데이터셋 로드 — 폴더 이름이 레이블이 됨 (neg=0, pos=1)
train_ds = keras.utils.text_dataset_from_directory("aclImdb/train", batch_size=batch_size)
val_ds   = keras.utils.text_dataset_from_directory("aclImdb/val",   batch_size=batch_size)
test_ds  = keras.utils.text_dataset_from_directory("aclImdb/test",  batch_size=batch_size)

# adapt() 에 쓸 "텍스트만" 데이터셋
text_only_train_ds = train_ds.map(lambda x, y: x)

# 텍스트 벡터화 설정
max_length = 600      # 문장을 600 토큰으로 자르거나 패딩
max_tokens = 20000    # 상위 20000 단어만 vocabulary 로 유지 (나머지는 OOV)

text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",                   # 정수 시퀀스로 인코딩
    output_sequence_length=max_length,   # 모든 배치에서 고정 길이
)
# adapt: train 데이터에서 vocabulary 구축 (단어 → 정수 매핑)
text_vectorization.adapt(text_only_train_ds)

# 데이터셋을 벡터화된 형태로 변환
int_train_ds = train_ds.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_val_ds   = val_ds.map(  lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_test_ds  = test_ds.map( lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)


## [6-2] Transformer Encoder 레이어 정의

### 내부 구성
1. **Positional Embedding** — 각 위치(0, 1, ..., max_length-1) 에 학습가능 vector 를 더함
2. **Multi-Head Attention** — 시퀀스 내 모든 토큰이 서로를 attention
3. **LayerNorm #1** — attention 결과에 residual 을 더한 뒤 정규화
4. **Dense Projection (FFN)** — Dense(dense_dim, relu) → Dense(embed_dim)
5. **LayerNorm #2** — FFN 결과에 residual 을 더한 뒤 정규화

### 왜 build() 를 오버라이드?
- Keras 는 레이어의 입력 shape 을 처음 call 할 때 알 수 있음
- Multi-Head Attention 같이 "shape 에 의존하는 sub-layer" 는 `build()` 에서 명시적으로 초기화해야 직렬화(save/load) 가 안정적
- `get_config()` 도 같은 이유 — 모델 저장 후 복원 시 이 클래스의 인자들을 재현하기 위해

### `attention_mask` 는 왜?
- 패딩 토큰 (실제 단어가 아닌 0) 은 attention 에서 제외해야 함
- Keras 의 Embedding 이 `mask_zero=True` 이면 mask 가 자동 전파됨 (여기선 기본 False 라 명시적 mask 는 None)


In [ ]:
# [6-2] Transformer Encoder + Positional Embedding (한 블록)
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, max_length, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim  = embed_dim
        self.dense_dim  = dense_dim
        self.num_heads  = num_heads
        self.max_length = max_length

        # Self-attention: 각 head 는 embed_dim 차원의 Q/K/V 를 계산
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim,
        )

        # FFN: 입력을 dense_dim 으로 확장했다가 embed_dim 으로 복원
        # 차원을 늘렸다 줄이는 패턴 — 표현력 확보
        self.dense_proj = keras.Sequential([
            layers.Dense(dense_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])

        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()

        # Positional Embedding — 위치 → vector
        # "어느 자리 인가" 라는 정보를 학습 가능한 형태로 주입
        self.position_embedding = layers.Embedding(
            input_dim=max_length, output_dim=embed_dim,
        )

    def build(self, input_shape):
        # input_shape = (batch, seq_length, embed_dim)
        seq_length = input_shape[1]
        self.position_embedding.build((seq_length,))

        # 내부 sub-layer 를 미리 build — save/load 안정성
        self.attention.build(
            query_shape=(input_shape[0], seq_length, self.embed_dim),
            key_shape  =(input_shape[0], seq_length, self.embed_dim),
            value_shape=(input_shape[0], seq_length, self.embed_dim),
        )
        self.dense_proj.build((input_shape[0], seq_length, self.embed_dim))
        self.layernorm_1.build((input_shape[0], seq_length, self.embed_dim))
        self.layernorm_2.build((input_shape[0], seq_length, self.embed_dim))
        super().build(input_shape)

    def call(self, inputs, mask=None):
        # inputs: (batch, seq_length, embed_dim)
        seq_length = tf.shape(inputs)[1]

        # 0, 1, 2, ..., seq_length-1 을 위치 임베딩으로 변환
        positions = tf.range(start=0, limit=seq_length, delta=1)   # (seq_length,)
        embedded_positions = self.position_embedding(positions)     # (seq_length, embed_dim)
        inputs += embedded_positions                                # 위치 정보 주입 (broadcast)

        if mask is not None:
            mask = mask[:, tf.newaxis, :]                           # attention 용 shape 변환

        # 1) Self-Attention
        attention_output = self.attention(inputs, inputs, attention_mask=mask)
        # Residual + LayerNorm (Post-Norm 방식)
        proj_input = self.layernorm_1(inputs + attention_output)

        # 2) FFN
        proj_output = self.dense_proj(proj_input)
        # Residual + LayerNorm 두 번째
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        # 모델 save 후 load 시 __init__ 에 넘겨줄 인자를 돌려줌
        config = super().get_config()
        config.update({
            "embed_dim":  self.embed_dim,
            "num_heads":  self.num_heads,
            "dense_dim":  self.dense_dim,
            "max_length": self.max_length,
        })
        return config


## [6-3] 전체 모델 + 학습

### 하이퍼파라미터
- `embed_dim=256` — 각 토큰을 256차원 벡터로 표현 (BERT-base 는 768, 여기선 소형)
- `num_heads=2` — attention head 수
- `dense_dim=32` — FFN 중간 차원 (일반적으로 embed_dim*2~4 권장이지만 여기선 작게)

### 왜 GlobalMaxPooling1D?
- 시퀀스 (batch, seq_length, embed_dim) → (batch, embed_dim) 로 축소
- 각 차원별로 **시퀀스 내 최댓값** 을 뽑음 → "이 문장에서 가장 강한 긍/부정 신호"
- BERT 는 보통 CLS 토큰을 쓰지만, 여기선 단순 pooling 으로 충분

### 왜 sigmoid + binary_crossentropy?
- 이진분류 (pos/neg) → 1차원 확률만 필요 (softmax 불필요)
- sigmoid 는 단일 확률값을 [0,1] 로, binary_crossentropy 는 이에 대응하는 loss


In [ ]:
# [6-3] 전체 모델 조립 + 학습
vocab_size = 20000    # 단어 집합 크기 (adapt 단계에서 맞춤)
embed_dim  = 256
num_heads  = 2
dense_dim  = 32

inputs  = keras.Input(shape=(None,), dtype="int64")   # 가변 길이 허용
x       = layers.Embedding(vocab_size, embed_dim)(inputs)  # (batch, seq, 256)
x       = TransformerEncoder(embed_dim, dense_dim, num_heads, max_length=max_length)(x)
x       = layers.GlobalMaxPooling1D()(x)              # 시퀀스 → 벡터
x       = layers.Dropout(0.5)(x)                      # 과적합 방지
outputs = layers.Dense(1, activation="sigmoid")(x)    # pos=1, neg=0

model = keras.Model(inputs, outputs)
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()

# 학습 — 커스텀 레이어를 쓰기 때문에 load 할 때 custom_objects 필요
callbacks = [
    keras.callbacks.ModelCheckpoint("transformer_encoder.keras", save_best_only=True),
]
model.fit(int_train_ds, validation_data=int_val_ds, epochs=20, callbacks=callbacks)

# 체크포인트 재로드 (최적 epoch 의 모델)
model = keras.models.load_model(
    "transformer_encoder.keras",
    custom_objects={"TransformerEncoder": TransformerEncoder},
)
print(f"테스트 정확도: {model.evaluate(int_test_ds)[1]:.3f}")
